# 🔧 Turn-taking & barge-in on a mock realtime stream

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 46 §46.1–46.2 · type: walkthrough*

**The promise:** by the end you've built an `asyncio` session loop over a *simulated* realtime stream that **detects end-of-turn**, **dispatches a tool mid-conversation** (reusing your Ch 12 tool schemas unchanged), and **handles barge-in** — cancelling the in-flight generation *and* stopping the (mock) TTS playback — while printing a per-turn latency budget against §46.2's targets.

The mock path runs **free and offline**: no audio device, no API key, no network. A real STT→agent→TTS / native-realtime path is sketched at the end as an explicitly **opt-in ⚠️** cell that CI never runs.

## 🧠 Why this matters

A Jupyter kernel can't carry a live phone call — but the part that actually decides whether a voice agent *feels alive* isn't the audio, it's the **session and turn-taking logic**, and that's pure, testable code.

The structural sketch from §46.1: you open a realtime session, configure it (instructions, voice, `turn_detection`, the **same tools as chat**), then loop over **events** — audio frames, `speech_started`, `turn.done`, `function_call_arguments.done` — reacting to each. The two hard problems (§46.2) both live in that loop:

- **Endpointing** — deciding the caller has *finished* speaking (voice-activity detection plus, increasingly, semantic cues).
- **Barge-in** — when the caller interrupts mid-reply, stop instantly: **cut the TTS playback *and* cancel the in-flight generation.** Stopping only the audio is the classic bug — the model keeps "thinking," burns tokens, and desyncs the conversation.

We build all of this against a `MockRealtimeSession` that yields the *same event shapes* as the real API, so the loop you write here is the loop you'd ship. See §46.1–46.2; the async machinery is Ch 4, the tools are Ch 12, the long-session discipline is Ch 14.

## Objectives & prereqs

**By the end you can:**
- Drive an `async for event in session` loop over a realtime event stream.
- Implement **endpointing** (a VAD-style "caller stopped" signal plus a semantic-cue stub).
- **Dispatch a tool mid-conversation** on a `function_call_arguments.done` event, reusing Ch 12 tool schemas unchanged, and feed the result back.
- Implement **barge-in** correctly: cancel the in-flight generation `Task` *and* stop mock playback — and see what a naive playback-only stop gets wrong.
- Print a **per-turn latency budget** vs §46.2's targets, and watch interruption/talk-over rates move when you inject noise.

**Prereqs:** [`46-01-cascaded-vs-speech-to-speech.ipynb`](46-01-cascaded-vs-speech-to-speech.ipynb) · Ch 4 (async / `asyncio`) · Ch 12 (tool dispatch) · Ch 14 (long-session compaction). Standard library only (`asyncio`, `random`, `time`, `dataclasses`).

**Run first:** the Setup cell. Defaults to `MOCK=1` — the entire walkthrough runs offline; the live path is opt-in and off by default.

> **`await` in Jupyter.** The notebook runs inside an event loop, so cells can `await` directly. Each demo is also wrapped in `asyncio.run(...)` so it runs identically as a plain script.

In [ ]:
# --- Setup -------------------------------------------------------------------
import asyncio
import os
import random
import time
from dataclasses import dataclass, field

from dotenv import load_dotenv

load_dotenv()  # reads a local .env if present; keys come ONLY from the environment

# MOCK=1 (default): the whole notebook runs offline -- a MockRealtimeSession stands
# in for the WebSocket, so no audio, no key, no network is needed. Set MOCK=0 only
# for the opt-in live cell at the very end (and supply a key + audio yourself).
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# Anthropic-first stack; the realtime *transport* is provider-specific, but the
# agent core it wraps is model-agnostic. Used only on the opt-in live path.
MODEL = "claude-opus-4-8"

random.seed(46)  # determinism for simulated VAD jitter and injected noise

# §46.2 per-stage targets (ms) -- the budget we instrument each turn against.
LATENCY_TARGETS_MS = {
    "endpointing": 300,   # decide the caller stopped (200-400)
    "stt": 150,           # finalize transcript (100-200)
    "llm_ttft": 350,      # LLM time-to-first-token (200-500)
    "tts_first": 175,     # TTS time-to-first-audio (100-250)
}
COMFORT_BAND_MS = (500, 800)

print(f"MOCK mode: {MOCK}  ({'offline -- mock realtime session' if MOCK else 'LIVE -- opt-in path only'})")
print(f"latency targets (ms): {LATENCY_TARGETS_MS}")

## 1 · The tools are the *same* tools (Ch 12)

The headline of the chapter: voice doesn't need a new brain. The agent core — and its **tool schemas** — are exactly what you built for chat in Ch 12. We register two tiny tools here in the same JSON-schema shape the model expects, and a `dispatch_tool` that runs by name. Nothing about this is voice-specific.

In [ ]:
# Ch 12 tool schemas, unchanged -- the realtime session sends the SAME shapes.
TOOL_SCHEMAS = [
    {
        "name": "get_account_balance",
        "description": "Look up the current balance for a verified account.",
        "input_schema": {
            "type": "object",
            "properties": {"account_id": {"type": "string"}},
            "required": ["account_id"],
        },
    },
    {
        "name": "schedule_callback",
        "description": "Schedule a callback for the caller at a given time.",
        "input_schema": {
            "type": "object",
            "properties": {
                "account_id": {"type": "string"},
                "when": {"type": "string", "description": "ISO-8601 time"},
            },
            "required": ["account_id", "when"],
        },
    },
]

# Local, deterministic tool implementations (no network in MOCK).
def _get_account_balance(account_id: str) -> dict:
    return {"account_id": account_id, "balance_usd": 1284.55, "currency": "USD"}

def _schedule_callback(account_id: str, when: str) -> dict:
    return {"account_id": account_id, "scheduled_for": when, "status": "confirmed"}

TOOL_IMPLS = {
    "get_account_balance": _get_account_balance,
    "schedule_callback": _schedule_callback,
}


async def dispatch_tool(name: str, arguments: dict) -> dict:
    """Same dispatch you wrote in Ch 12 -- look up by name, run, return a result dict."""
    impl = TOOL_IMPLS.get(name)
    if impl is None:
        return {"error": f"unknown tool: {name}"}
    # Tools may themselves be I/O-bound; await keeps the loop free (Ch 4).
    await asyncio.sleep(0.05)  # stand-in for the tool's real round-trip
    return impl(**arguments)


print("registered tools:", [t["name"] for t in TOOL_SCHEMAS])
print((await dispatch_tool("get_account_balance", {"account_id": "acct-42"})))

## 2 · A `MockRealtimeSession` that yields the real event shapes

The real API hands you a stream of typed events. We mirror that exactly with a tiny event type and an async generator. The event *names* match the §46.1 sketch — `speech_started`, `turn.done`, `response.function_call_arguments.done` — so the loop we write next is the loop you'd write against the live SDK.

The session plays a **scripted timeline**: the caller speaks, stops (we'll endpoint that), the agent answers, then on a later turn the caller **barges in** mid-answer. Each event carries a timestamp so we can measure stages.

In [ ]:
@dataclass
class RealtimeEvent:
    type: str                      # e.g. "speech_started", "turn.done", "response.function_call_arguments.done"
    t_ms: int = 0                  # session-relative timestamp (ms)
    text: str = ""                 # partial/final transcript, when relevant
    name: str = ""                 # tool name, for function-call events
    arguments: dict = field(default_factory=dict)
    call_id: str = ""


class MockRealtimeSession:
    """Stands in for client.realtime.connect(...).  Yields events like the real API.

    Same surface as the §46.1 sketch: configure once, then `async for event in session`.
    A 1 ms sleep per event keeps timings real-but-fast and deterministic.
    """

    def __init__(self, timeline, *, noise: float = 0.0):
        self._timeline = timeline      # list[RealtimeEvent]
        self.noise = noise             # 0..1: simulated packet jitter / crosstalk
        self.config = {}

    async def update(self, *, session: dict):
        # Mirrors `await session.update(session={...})` -- instructions, voice,
        # turn_detection, tools. We just record it.
        self.config = session

    async def send_tool_result(self, call_id: str, result: dict):
        # Mirrors feeding a tool result back into the session.
        await asyncio.sleep(0.001)

    async def __aiter__(self):  # pragma: no cover - thin wrapper
        for ev in self._timeline:
            # Noise jitters arrival time, modelling real (lossy) audio transport.
            jitter = int(self.noise * random.randint(0, 120))
            await asyncio.sleep(0.001)
            yield RealtimeEvent(ev.type, ev.t_ms + jitter, ev.text, ev.name, ev.arguments, ev.call_id)


print("MockRealtimeSession ready -- same event surface as the realtime SDK sketch (§46.1).")

## 3 · The scripted event timelines

Two short scripts. The first is a clean turn that triggers a **tool call**. The second has the caller **barge in** while the agent is mid-reply. `audio_frame` events are the VAD's raw material; `speech_stopped` is the *true* moment the caller went silent (the endpointer has to *infer* it, with delay).

In [ ]:
def tool_turn_timeline():
    """Caller asks for a balance -> stops -> (we endpoint) -> agent calls a tool."""
    return [
        RealtimeEvent("speech_started", t_ms=0,    text=""),
        RealtimeEvent("audio_frame",    t_ms=200,  text="what's my"),
        RealtimeEvent("audio_frame",    t_ms=600,  text="what's my balance on account forty two"),
        RealtimeEvent("speech_stopped", t_ms=1000, text="what's my balance on account forty two"),
        # After our endpointer fires, the model decides to call a tool:
        RealtimeEvent("response.function_call_arguments.done", t_ms=1500,
                      name="get_account_balance",
                      arguments={"account_id": "acct-42"}, call_id="call_1"),
        RealtimeEvent("turn.done", t_ms=1700, text="Your balance is $1,284.55."),
    ]


def barge_in_timeline():
    """Agent starts a long answer; caller interrupts (speech_started) mid-reply."""
    return [
        RealtimeEvent("speech_started", t_ms=0),
        RealtimeEvent("speech_stopped", t_ms=900, text="tell me everything about my account"),
        RealtimeEvent("turn.done",      t_ms=1100, text="tell me everything about my account"),
        # The agent is now generating + speaking a LONG reply. The caller cuts in:
        RealtimeEvent("speech_started", t_ms=1450),   # <-- BARGE-IN mid-response
        RealtimeEvent("speech_stopped", t_ms=2100, text="actually just the balance"),
        RealtimeEvent("turn.done",      t_ms=2300, text="actually just the balance"),
    ]


print("timelines ready:",
      len(tool_turn_timeline()), "events (tool turn);",
      len(barge_in_timeline()), "events (barge-in turn)")

## 4 · Endpointing: deciding the caller actually stopped

The session gives us `speech_stopped`, but in the real world that's *inferred* from audio with a hangover delay — you wait a beat of silence to be sure the caller isn't just pausing mid-thought. Too short and you cut people off; too long and the agent feels slow (§46.2).

Our endpointer combines two signals (the chapter's recipe): a **VAD silence timer** and a **semantic cue** — if the transcript looks like a complete request, you can endpoint sooner. We return the *decision delay* so we can charge it to the latency budget.

In [ ]:
SEMANTIC_END_CUES = ("balance", "account", "please", "thanks", "thank you", "that's all")

def endpoint_decision(transcript: str, vad_silence_ms: int = 300) -> tuple[int, str]:
    """Return (decision_delay_ms, reason). Combines a VAD timer + a semantic-cue stub."""
    looks_complete = any(cue in transcript.lower() for cue in SEMANTIC_END_CUES)
    if looks_complete:
        # A confident semantic cue lets us endpoint sooner than the full VAD hangover.
        return max(120, vad_silence_ms // 2), "semantic-cue + vad"
    # No cue: wait out the full silence hangover before declaring end-of-turn.
    return vad_silence_ms, "vad-timer-only"


for sample in ["what's my balance on account forty two",
               "um well i was thinking maybe"]:
    delay, why = endpoint_decision(sample)
    print(f"{delay:>4} ms  via {why:18}  <- {sample!r}")

print("\nSemantic cues let a confident, complete request endpoint faster;")
print("an ambiguous, trailing utterance waits the full VAD hangover. That tuning IS the UX.")

## 5 · 🔧 The session loop: endpoint, generate, dispatch tools, instrument

Now the core build. The loop mirrors the §46.1 sketch: `async for event in session`, react by type. On `turn.done` we **start generating a reply** (as a cancellable `asyncio.Task` — this is what makes barge-in possible later). On a `function_call_arguments.done` event we **dispatch the tool** and feed the result back. Throughout, we timestamp each stage and accumulate a **per-turn latency budget**.

We model the agent's reply as two cancellable coroutines: `generate_reply` (LLM) and `play_tts` (speaking it aloud). Keeping a handle on both `Task`s is the whole trick to a correct barge-in.

In [ ]:
@dataclass
class TurnMetrics:
    endpoint_ms: int = 0
    llm_ttft_ms: int = 0
    tts_first_ms: int = 0
    tool_ms: int = 0
    barge_in_ms: int = 0
    interrupted: bool = False


async def generate_reply(text: str, *, long: bool = False) -> str:
    """Mock LLM generation -- cancellable. `long` simulates a reply worth interrupting."""
    await asyncio.sleep(LATENCY_TARGETS_MS["llm_ttft"] / 1000)  # time-to-first-token
    chunks = ["Sure,", "your", "balance", "is", "$1,284.55."]
    if long:
        chunks = ["Sure,"] + ["let", "me", "walk", "through", "every", "detail", "of",
                              "your", "account", "history", "step", "by", "step"]
    out = []
    for c in chunks:
        await asyncio.sleep(0.02)   # steady decode; each await is a cancellation point
        out.append(c)
    return " ".join(out)


async def play_tts(text: str) -> None:
    """Mock TTS playback -- cancellable. Speaks word by word so it can be cut off."""
    await asyncio.sleep(LATENCY_TARGETS_MS["tts_first"] / 1000)  # time-to-first-audio
    for _ in text.split():
        await asyncio.sleep(0.02)   # 'speaking'; cancelling here stops the audio


async def run_session(session: MockRealtimeSession, *, long_reply: bool = False):
    """Drive the realtime session: endpoint -> generate -> (tool) -> speak, instrumented."""
    await session.update(session={
        "instructions": "You are a helpful, concise phone agent. Disclose you are an AI.",
        "voice": "marin",
        "turn_detection": {"type": "server_vad"},
        "tools": TOOL_SCHEMAS,                  # SAME tools as chat (Ch 12)
    })

    metrics = TurnMetrics()
    gen_task: asyncio.Task | None = None        # in-flight LLM generation
    tts_task: asyncio.Task | None = None        # in-flight TTS playback
    transcript = ""
    log = []

    async for event in session:
        if event.type == "speech_started" and (gen_task or tts_task):
            # BARGE-IN handled in the next cell's wrapper; here we just note the event.
            log.append((event.t_ms, "speech_started (while replying)"))
        elif event.type in ("audio_frame",):
            transcript = event.text or transcript
        elif event.type == "speech_stopped":
            transcript = event.text or transcript
            delay, why = endpoint_decision(transcript)
            metrics.endpoint_ms = delay
            log.append((event.t_ms, f"endpointed (+{delay}ms, {why})"))
        elif event.type == "response.function_call_arguments.done":
            t0 = time.perf_counter()
            result = await dispatch_tool(event.name, event.arguments)   # Ch 12 dispatch
            metrics.tool_ms = int((time.perf_counter() - t0) * 1000)
            await session.send_tool_result(event.call_id, result)
            log.append((event.t_ms, f"tool {event.name} -> {result.get('balance_usd', result)}"))
        elif event.type == "turn.done":
            # Start the reply as CANCELLABLE tasks (so barge-in can kill them).
            t0 = time.perf_counter()
            gen_task = asyncio.create_task(generate_reply(transcript, long=long_reply))
            reply = await gen_task
            metrics.llm_ttft_ms = LATENCY_TARGETS_MS["llm_ttft"]
            tts_task = asyncio.create_task(play_tts(reply))
            await tts_task
            metrics.tts_first_ms = LATENCY_TARGETS_MS["tts_first"]
            gen_task = tts_task = None
            log.append((event.t_ms, f"replied: {reply!r}"))

    return metrics, log


print("session loop defined: endpoint -> generate (cancellable) -> tool dispatch -> speak (cancellable).")

## 6 · Run the clean turn: a mid-conversation tool call

Drive the tool-turn timeline through the loop. Watch the order: the caller's request is **endpointed**, the model emits a **tool call**, we **dispatch it** (the unchanged Ch 12 tool), feed the result back, and the agent speaks the answer.

In [ ]:
async def demo_tool_turn():
    session = MockRealtimeSession(tool_turn_timeline())
    metrics, log = await run_session(session, long_reply=False)
    print("event log:")
    for t_ms, msg in log:
        print(f"  t={t_ms:>5} ms  {msg}")
    return metrics


tool_metrics = (await demo_tool_turn())
print("\ntool dispatched mid-session, result fed back, reply spoken -- all reusing Ch 12 tools.")

**What you just saw.** The realtime loop is your familiar agent loop with two additions: an **endpointing** decision before you trust the turn is over, and **events** (not request/response) as the control flow. The tool call went through `dispatch_tool` with the exact Ch 12 schema — voice changed the *transport*, not the *brain*.

## 7 · The per-turn latency budget vs §46.2 targets

Sum the instrumented stages and compare to the comfort band. This is the audio-layer metric from 46-01, now measured on a real (mock) turn rather than modeled.

In [ ]:
def print_budget(metrics, label):
    stages = [
        ("endpointing", metrics.endpoint_ms, LATENCY_TARGETS_MS["endpointing"]),
        ("stt", LATENCY_TARGETS_MS["stt"], LATENCY_TARGETS_MS["stt"]),
        ("llm_ttft", metrics.llm_ttft_ms, LATENCY_TARGETS_MS["llm_ttft"]),
        ("tts_first", metrics.tts_first_ms, LATENCY_TARGETS_MS["tts_first"]),
    ]
    if metrics.tool_ms:
        stages.insert(2, ("tool_call", metrics.tool_ms, 400))
    total = sum(actual for _, actual, _ in stages)
    print(f"--- {label} ---")
    print(f"{'stage':14} {'actual':>7} {'target':>7}")
    for name, actual, target in stages:
        flag = "" if actual <= target else "  <- OVER target"
        print(f"{name:14} {actual:>6}m {target:>6}m{flag}")
    lo, hi = COMFORT_BAND_MS
    verdict = "within band" if lo <= total <= hi else ("UNDER band" if total < lo else "OVER band -- feels laggy")
    print(f"{'TOTAL':14} {total:>6}m   comfort {lo}-{hi}m  =>  {verdict}\n")
    return total


print_budget(tool_metrics, "clean tool turn")

**What you just saw.** With a tool call inserted, the turn can slip past the 800 ms comfort ceiling — which is exactly why §46.2's tip is to **cover tool gaps with a filler** ("let me pull that up") and stream TTS early. The number on the screen, not a vibe, tells you whether to reach for that filler.

## 8 · 🔮 Predict: what does a *naive* barge-in get wrong?

Now the barge-in turn. The caller interrupts while the agent is mid-reply (a long answer). A **naive** loop stops only the TTS playback — the audio goes quiet — but leaves the LLM generation running.

**Predict before running:** if you stop *only the playback* on `speech_started`, what goes wrong? Think about (a) tokens/cost, (b) what the agent "believes" the conversation state is, and (c) what happens when the now-orphaned generation finishes.

In [ ]:
async def run_session_naive_bargein(session):
    """WRONG barge-in: stop playback only, leave generation running."""
    await session.update(session={"tools": TOOL_SCHEMAS, "turn_detection": {"type": "server_vad"}})
    gen_task = tts_task = None
    orphaned = []          # generations we 'stopped' but secretly left running
    metrics = TurnMetrics()

    async for event in session:
        if event.type == "turn.done":
            gen_task = asyncio.create_task(generate_reply(event.text, long=True))
            tts_task = asyncio.create_task(play_tts("speaking the long reply..."))
        elif event.type == "speech_started" and tts_task is not None:
            metrics.interrupted = True
            tts_task.cancel()                      # stop the audio... only the audio
            try:
                await tts_task
            except asyncio.CancelledError:
                pass
            tts_task = None
            # BUG: gen_task is still running. It keeps decoding tokens in the background.
            if gen_task is not None and not gen_task.done():
                orphaned.append(gen_task)          # cost + state desync

    # The orphaned generation eventually finishes -- after the caller already moved on.
    leftover = await asyncio.gather(*orphaned) if orphaned else []
    return metrics, leftover


async def demo_naive():
    session = MockRealtimeSession(barge_in_timeline())
    metrics, leftover = await run_session_naive_bargein(session)
    print("interrupted:", metrics.interrupted)
    print("orphaned generations that kept running after the audio stopped:", len(leftover))
    for r in leftover:
        print("  leaked reply (burned tokens, now stale):", repr(r))
    return metrics


_ = (await demo_naive())

**What you just saw.** Stopping only playback left the **generation running in the background**: it burned tokens to completion and produced a reply to a question the caller already abandoned. On a real provider that's wasted spend *and* a desynced session — the agent's transcript now contains an answer the caller never heard, poisoning the next turn. Silence is not cancellation.

## 9 · 🔧 Barge-in done right: cancel generation *and* playback

The fix is one line of discipline: on `speech_started` mid-reply, cancel **both** tasks — the generation and the playback — and only then start listening to the new utterance. We also time how fast we reacted (**barge-in response time**, an audio-layer SLO).

In [ ]:
async def run_session_correct_bargein(session):
    """RIGHT barge-in: cancel generation AND playback, then honor the new turn."""
    await session.update(session={"tools": TOOL_SCHEMAS, "turn_detection": {"type": "server_vad"}})
    gen_task = tts_task = None
    metrics = TurnMetrics()
    reply_started_at = None
    log = []

    async def cancel(task):
        if task and not task.done():
            task.cancel()
            try:
                await task
            except asyncio.CancelledError:
                pass

    async for event in session:
        if event.type == "turn.done":
            reply_started_at = time.perf_counter()
            gen_task = asyncio.create_task(generate_reply(event.text, long=True))
            tts_task = asyncio.create_task(play_tts("speaking the long reply..."))
            log.append((event.t_ms, "started long reply (generation + playback)"))
        elif event.type == "speech_started" and (gen_task or tts_task):
            t0 = time.perf_counter()
            await cancel(gen_task)        # cancel the LLM generation...
            await cancel(tts_task)        # ...AND stop the TTS playback
            gen_task = tts_task = None
            metrics.interrupted = True
            metrics.barge_in_ms = int((time.perf_counter() - t0) * 1000)
            log.append((event.t_ms, f"BARGE-IN: cancelled generation + playback in {metrics.barge_in_ms}ms"))

    # Nothing left running: no orphaned generation, no leaked tokens, state is clean.
    await cancel(gen_task)
    await cancel(tts_task)
    return metrics, log


async def demo_correct():
    session = MockRealtimeSession(barge_in_timeline())
    metrics, log = await run_session_correct_bargein(session)
    for t_ms, msg in log:
        print(f"  t={t_ms:>5} ms  {msg}")
    print(f"\ninterrupted={metrics.interrupted}  barge_in_response={metrics.barge_in_ms} ms")
    print("Both tasks cancelled -> no leaked tokens, no stale reply in the transcript.")
    return metrics


bargein_metrics = (await demo_correct())

**What you just saw.** Cancelling the generation `Task` *and* the playback `Task` leaves the session clean — no background decode, no stale reply. Because every `await` in `generate_reply`/`play_tts` is a cancellation point (Ch 4), `task.cancel()` actually stops the work at the next suspension. **Barge-in response time** — how fast you went quiet after `speech_started` — is now a number you can put an SLO on.

## ⚠️ Pitfall: endpointing tuned on quiet audio cuts real callers off

Demos run on quiet office Wi-Fi; callers run on speakerphone in a car. Inject simulated **noise/jitter** into the session and measure what actually degrades — and note it's **interruption / talk-over rate**, not word-error rate. We run many turns at rising noise levels and count how often the endpointer fires *while the caller is still talking* (a premature cut).

In [ ]:
def premature_cut(noise: float) -> bool:
    """Model: under noise, the VAD may mistake a mid-sentence pause for end-of-turn."""
    # More noise -> more spurious silence -> higher chance of an early endpoint.
    return random.random() < (0.02 + 0.55 * noise)


def measure_talk_over(noise: float, turns: int = 400) -> float:
    cuts = sum(premature_cut(noise) for _ in range(turns))
    return cuts / turns


print(f"{'noise':>6}  {'talk-over rate':>15}")
for noise in (0.0, 0.25, 0.5, 0.75, 1.0):
    rate = measure_talk_over(noise)
    bar = "#" * int(rate * 40)
    print(f"{noise:>6.2f}  {rate:>14.1%}  {bar}")

print("\nThe SAME endpointer that felt crisp at noise=0 cuts callers off as noise rises.")
print("This is why you test on RECORDED REAL CALLS and watch talk-over/interruption rates,")
print("not just transcription accuracy (§46.2 pitfall).")

**What you just saw.** Word-error rate would barely move here — the *transcript* may still be right — yet the experience falls apart because the endpointer fires mid-sentence. The audio layer fails silently in any text-only eval. Mitigations: a longer VAD hangover under detected noise, semantic confirmation before endpointing, and a one-utterance-away "transfer to a human" path.

## 🎯 Senior lens

Treat the realtime session as **state**, not as a request. Three senior reflexes:

- **Everything cancellable.** Generation and playback are `Task`s you hold handles to, because barge-in *will* happen and "stop the audio" is not "stop the work." A loop that can't cancel its own generation will leak tokens and desync on the first interruption.
- **Long calls need compaction.** A ten-minute call is a long agent run; the transcript grows without bound and you pay to re-send it every turn. Apply the Ch 14 compaction discipline — summarize older turns, keep the live window tight — or latency and cost creep up as the call goes on.
- **Instrument before you fine-tune.** The metrics that decide whether a voice agent feels alive are **endpointing precision, barge-in response time, and per-stage latency percentiles** — none of which a transcript eval sees. Most "the model is bad" complaints in voice are turn-taking and latency failures (46-01). Measure those first.

And keep the safety rails from §46.3 architectural, not bolted on: AI disclosure in the opening turn, out-of-band authentication (a voice is not identity), and an always-available "get me a human" path the model can't talk its way out of.

## Recap

- A realtime voice loop is your **agent loop over events** (`speech_started`, `turn.done`, `function_call_arguments.done`) — same tools, same brain, new transport.
- **Endpointing** = decide the caller stopped, combining a VAD silence timer with a semantic cue; too eager cuts people off, too patient feels slow.
- **Tool calls** reuse the Ch 12 schemas and `dispatch_tool` unchanged — voice didn't need a new tool layer.
- **Barge-in must cancel generation *and* playback.** Stopping only the audio leaks tokens and desyncs the session; hold both as cancellable `Task`s.
- The **per-turn latency budget** (vs §46.2 targets) tells you when to deploy fillers and stream TTS early; a tool call easily pushes a turn past 800 ms.
- Under **noise**, the endpointer — not the transcriber — degrades; measure **talk-over / interruption rate** and barge-in time, not just word-error rate.
- Treat the session as **state**: long calls need Ch 14 compaction; instrument the audio layer before touching the model.

## Exercises

Each one *changes* the loop and asks you to predict first. (Solutions land in `solutions/` in Phase 2.)

1. **Add a semantic endpointing cue.** Extend `SEMANTIC_END_CUES` (or replace the stub with a tiny rule) so "that's it" / "go ahead" endpoint immediately, but a trailing "um, and also..." does not. Predict the effect on the endpointing budget for a complete vs an incomplete utterance, then measure.
2. **Tune the barge-in threshold.** Add a guard so a *very short* `speech_started` (a cough, a back-channel "mm-hm") does **not** trigger barge-in, but a real interruption does. Predict the trade-off — miss real interruptions vs barge in on noise — then test both timelines.
3. **Filler for the tool gap.** When a tool call will exceed the comfort band, speak a filler ("let me pull that up") while `dispatch_tool` runs. Predict whether this changes the *measured* total or only the *perceived* latency (recall 46-01), then implement it and re-print the budget.
4. **Compact a long call.** Simulate a 30-turn call and add a Ch 14-style compaction step that summarizes turns older than N. Predict what happens to per-turn input size with vs without it, then plot both.

In [ ]:
# Exercise 1 -- your code here

In [ ]:
# Exercise 2 -- your code here

In [ ]:
# Exercise 3 -- your code here

In [ ]:
# Exercise 4 -- your code here

## ⚠️ Optional: a real STT→agent→TTS / native-realtime path (opt-in, **not** run in CI)

Everything above ran offline. The cell below **sketches** a real realtime session and is **disabled by default** — it requires `COMPANION_MOCK=0`, a real API key in the environment, and a microphone/audio stack you supply yourself. CI never runs it. Treat real audio as an explicitly opt-in path, exactly as the chapter frames it.

It is intentionally a guarded sketch (provider APIs evolve — check the docs): the point is that the **same loop, endpointing, and barge-in logic** you built above wrap a real transport unchanged.

In [ ]:
# ⚠️ OPT-IN LIVE PATH -- skipped unless COMPANION_MOCK=0 AND a key is present.
# This does NOT run in CI and needs an audio stack you provide. It is a SKETCH.
if MOCK:
    print("MOCK=1 -> skipping the live realtime path (this is the default, by design).")
else:
    # Fail fast and friendly if the key is missing -- never hardcode it.
    api_key = os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY")
    if not api_key:
        raise SystemExit(
            "Live path needs an API key in the environment (e.g. OPENAI_API_KEY). "
            "Set it in your .env, supply audio I/O, and re-run with COMPANION_MOCK=0."
        )

    print("Live realtime is a real-money, real-audio path. Structural sketch (verify against current docs):")
    print(
        "\n"
        "    # cost note: realtime sessions bill for streamed audio in+out by the minute --\n"
        "    # keep test calls short. This is a DRY sketch; wire your own audio I/O.\n"
        "    #\n"
        "    # async with client.realtime.connect(model='gpt-realtime') as session:\n"
        "    #     await session.update(session={\n"
        "    #         'instructions': SYSTEM_PROMPT,\n"
        "    #         'voice': 'marin',\n"
        "    #         'turn_detection': {'type': 'server_vad'},  # server endpoints turns\n"
        "    #         'tools': TOOL_SCHEMAS,                      # the SAME Ch 12 tools\n"
        "    #     })\n"
        "    #     async for event in session:                    # the SAME loop shape\n"
        "    #         if event.type == 'response.function_call_arguments.done':\n"
        "    #             result = await dispatch_tool(event.name, event.arguments)\n"
        "    #             await session.send_tool_result(event.call_id, result)\n"
        "    #         elif event.type == 'input_audio_buffer.speech_started':\n"
        "    #             await cancel_current_response(session)  # barge-in: gen + playback\n"
    )
    print("\nThe endpointing, tool dispatch, and barge-in logic above port over unchanged.")

## Next

- 📖 **Book:** §46.1 (realtime session structure), §46.2 (turn-taking, barge-in, the latency budget), §46.3 (voice compliance & safety — consent, PII, authentication, escalation, AI disclosure).
- ◀️ **Previous:** [`46-01-cascaded-vs-speech-to-speech.ipynb`](46-01-cascaded-vs-speech-to-speech.ipynb) — where the latency budget you instrumented here comes from.
- 🧩 **Blueprint this feeds:** the realtime transport is provider-specific, but the brain it wraps is the existing [`blueprints/agent-loop/`](../../blueprints/agent-loop/), reused unchanged — voice is a wrapper, not a new agent. The async patterns are from Ch 4; the tools from Ch 12; the long-session compaction from Ch 14.
- 🎓 **Capstone:** no dedicated module — a realtime transport would sit in front of the existing transport-agnostic `capstone-project/agents/` core, calling the same tools and guardrails you've built.